** Problem Statement**: Analyze the data and generate insights that could help Netflix in deciding which type of shows/movies to produce and how they can grow the business in different countries

## Importing Pckages and Data Set

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import gdown

import warnings
warnings.filterwarnings('ignore', category = DeprecationWarning)

In [ ]:
!gdown 1rMcDU1a4k0oJT34KIJLT_5BmZPkkhIHJ

In [ ]:
data = pd.read_csv('netflix.csv')

In [ ]:
data.head()

## Initial Data Analysis
### Data Description
This dataset contains Netflix titles including Movies and TV Shows, along with metadata such as country, release year, genre, rating, and duration.

### Data Shape and Size

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
# Keeping the original data intact and using a copy for further analysis
data_new = data.copy()

## Missing Data

In [ ]:
# Original dataset
data.isnull().sum()

In [ ]:
missing_percent = data.isnull().mean() * 100
print(missing_percent)

In [ ]:
# New dataset
data_new.isnull().sum()

In [ ]:
missing_percent = data_new.isnull().mean() * 100
print(missing_percent)

**Analysis**: Significant missing data in the columns director (29.9%), cast (9.37%), and country (9.45%). Low data chunk missing in date_added, rating, duration.

## Handling the missing values

In [ ]:
data_new.dtypes

### Handling of categorical Columns

In [ ]:
cols_to_handle = ['director', 'cast', 'country']

In [ ]:
# Converting to String

for col in cols_to_handle:
  data_new[col] = data_new[col].astype(str)

In [ ]:
# Removing trailing and leading spaces

for col in cols_to_handle:
  data_new[col] = data_new[col].str.strip()

In [ ]:
# Converting string nan to NA

for col in cols_to_handle:
  data_new[col] = data_new[col].replace('nan', pd.NA)

In [ ]:
# Replacing missing categorical values with "Unknown <Column>"

data_new['country'] = data_new['country'].fillna('Unknown Country')
data_new['cast'] = data_new['cast'].fillna('Unknown Cast')
data_new['director'] = data_new['director'].fillna('Unknown Director')

### Handling of missing data in rating column

In [ ]:
# Converting rating to string and removing trailing and leading spaces
data_new['rating'] = data_new['rating'].astype(str).str.strip()

# Converting string nan to NA
data_new['rating'] = data_new['rating'].replace('nan', pd.NA)

# Replacing missing rating values with "Unknown Rating"
data_new['rating'] = data_new['rating'].fillna('Unknown Rating')

# Converting rating column to category
data_new['rating'] = data_new['rating'].astype('category')

### Handling of missing data in duration column
Duration column comprises of numerical value and string

In [ ]:
# Extracting the numerical value from the column

data_new['duration'] = data_new['duration'].astype(str).str.strip()

Handling the movies data and tv shows differently

In [ ]:
# Movies

data_new['duration_minutes'] = (data_new.loc[data_new['type'] == 'Movie', 'duration'].str.replace(' min', '', regex=False))
data_new['duration_minutes'] = pd.to_numeric(data_new['duration_minutes'], errors='coerce')
data_new['duration_minutes'] = data_new['duration_minutes'].fillna(0)

In [ ]:
# TV Shows

data_new['seasons'] = (data_new.loc[data_new['type'] == 'TV Show', 'duration'].str.split(' ').str[0])
data_new['seasons'] = pd.to_numeric(data_new['seasons'], errors='coerce')
data_new['seasons'] = data_new['seasons'].fillna(0)

**Note:** The duration column actually means two things: for movies, it shows minutes, and for TV shows, it lists the number of seasons. So, instead of combining everything together as just one number, splitted the column into two new columns—duration_minutes for movies and seasons for TV shows. This way, the data stays accurate and avoid inaccurate treatment.

### Handling of missing data in date_added column

In [ ]:
# Converting date_added column to datetime for further analysis

data_new['date_added'] = pd.to_datetime(data_new['date_added'], errors='coerce')

In [ ]:
# Filling the missing values with the max date in the data (most recent date)

data_new['date_added'] = data_new['date_added'].fillna(data_new['date_added'].max())

**Note:** The date_added column was converted into three separate features: year, month, and day of addition. The transformation provides the ability to conduct time-based analysis which includes yearly content trends and monthly seasonality patterns and the determination of optimal periods for content release. The missing data was filled in before conducting the analysis to prevent any null values from affecting the creation of new columns.

In [ ]:
# Check if any missing values still exists in the data

data_new.info()

In [ ]:
data_new.isnull().sum()

## Un-nesting the columns

In [ ]:
# Columns to un-nest

cols_to_unnest = ['cast', 'director', 'country', 'listed_in']

In [ ]:
for col in cols_to_unnest:
    data_new[col] = data_new[col].str.split(',')

In [ ]:
# # Applying un-nesting on all columns

for col in cols_to_unnest:
    data_new = data_new.explode(col)

In [ ]:
# Resetting index after un-nesting
data_new = data_new.reset_index(drop=True)

**Note:** The dataFrame generated duplicate index labels after un-nesting the columns. The reset_index function with drop=True was applied to remove the duplicacy of indexes.

In [ ]:
# Check if un-nesting created any additional missing values in the dataset

data_new.info()

In [ ]:
data_new.isnull().sum()

This ensures our un-nesting has not created any additional missing values in the dataset

## Data Analysis
### Analysis of Categorical Columns
#### Rating Counts

In [ ]:
# For original dataset

data['rating'].value_counts().head(10)

In [ ]:
plt.figure(figsize = (8, 6))
sns.countplot(data = data, x = 'rating')
plt.xlabel('Ratings')
plt.ylabel('Count')
plt.xticks(rotation = 45)
plt.title('Rating Counts')
plt.show()

In [ ]:
perc_count_ratings = data['rating'].value_counts(normalize=True) * 100
perc_count_ratings

In [ ]:
# For new dataset

data_new['rating'].value_counts().head(10)
perc_count_ratings_new = data['rating'].value_counts(normalize=True) * 100
perc_count_ratings_new

In [ ]:
plt.figure(figsize = (8, 6))
sns.countplot(data = data_new, x = 'rating')
plt.xlabel('Ratings')
plt.ylabel('Count')
plt.xticks(rotation = 45)
plt.title('Rating Counts')
plt.show()

**Note:** Using top 10 rating count for further analysis as the graph after un-nesting and handling missing values has become more crowded

In [ ]:
# Top 10 ratings

top_10_ratings = data_new['rating'].value_counts().head(10).index
top_10_ratings

In [ ]:
plt.figure(figsize=(8,6))

sns.countplot(data = data_new, x = 'rating', order = top_10_ratings)
plt.xlabel('Ratings')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.title('Top 10 Netflix Ratings')
plt.show()

**Anslysis:** Based on the top 10 rating value counts after handling missing values and un-nesting, the most important ratings are TV-MA, TV-14, R, PG-13, TV-PG, and PG. This shows that Netflix has content is more driven towards mature, young adult, and general audiences.

#### Country Counts

In [ ]:
data['country'].value_counts().head(10)

In [ ]:
perc_count_country = data['country'].value_counts(normalize=True) * 100
perc_count_country

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x = data['country'])
plt.xlabel("Number of Titles Per Country")
plt.ylabel("Country")
plt.xticks(rotation = 45)
plt.show()

**Note:** Using Top 10 countries as it will be too crowded and overlapped using the complete data. The above graph is hard to interpret.

In [ ]:
# Top 10 countries

top_countries = data['country'].value_counts().head(10).index
top_countries

In [ ]:
plt.figure(figsize = (8,5))
sns.countplot(x = data['country'], order = top_countries)
plt.title("Top 10 Content Producing Countries on Netflix")
plt.xlabel("Number of Titles Per Country")
plt.ylabel("Country")
plt.xticks(rotation = 45)
plt.show()

In [ ]:
# Using new dataset and checking only top 10 countries
top_countries_new = data_new['country'].value_counts().head(10).index

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(y="country", data=data_new[data_new['country'].isin(top_countries_new)], order=top_countries_new)
plt.title("Top 10 Content Producing Countries")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.show()

**Analysis:** Most important countries for Netlfix are USA, India and UK. These countries are producing more than 50% of all Netflix's content.

#### Genre Counts

In [ ]:
# Number of unique genres in original dataset

data['listed_in'].nunique()

In [ ]:
# Number of unique genres in original dataset

data_new['listed_in'].nunique()

**Note:** The original dataset showed 514 distinct genre combinations because the listed_in field contained multiple genres which were stored in a single database cell. After un-nesting the data each record showed a single genre which produced 73 distinct genres.

In [ ]:
# Using new dataset and checking only top 10 genres

top_10_genres = data_new['listed_in'].value_counts().head(10).index
top_10_genres

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(y = 'listed_in', data=data_new, order=top_10_genres)
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
# plt.xticks(rotation = 45)
plt.title("Top 10 Genres on Netflix")
plt.show()

**Analysis:** Netlix has a wide variety of genres. Content of Netflix is highly diversified i.e. it offers content across many different genres, languages, countries and audience segments.

In [ ]:
data_new.head()

#### Director Counts

In [ ]:
data_new['director'].nunique()

In [ ]:
# Excluding the unknown director records and creating a new dataset of directors

data_dir = data_new[data_new['director'] != 'Unknown Director'].copy()
data_dir.head()

In [ ]:
# Using new dataset and checking only top 10 directors
top_10_directors = data_dir['director'].value_counts().head(10)
top_10_directors

In [ ]:
plt.figure(figsize=(8,6))

sns.countplot(y = 'director', data = data_dir, order = top_10_directors.index)
plt.xlabel("Number of Titles")
plt.ylabel("Director")
plt.title("Top 10 Directors on Netflix")
plt.show()

**Analysis:** The count plot clearly shows that a few number of directors contribute to the most number of movies/tv shows, indicating a dependence on repetitive directors. The Top 10 Directors identified in the dataset are associated exclusively with movies, which is why only 'Movie' appears in the legend. This indicates that directors who work frequently on Netflix tend to focus their efforts on filmmaking instead of producing television shows.

#### Directors Working on both movies and TV shows

In [ ]:
# Grouping on the basis of directors and type of content

director_type_check = (data_dir.groupby(['director', 'type']).size().reset_index(name='count'))
director_type_check

In [ ]:
# Checking directors name which are in top 10 list

director_type_check[director_type_check['director'].isin(top_10_directors.index)]

In [ ]:
# Top 10 directors making Tv shows

data_dir[data_dir['type'] == 'TV Show']['director'].value_counts().head(10)

In [ ]:
# Directors who worked on both movies and TV shows

both_types = (data_dir.groupby('director')['type'].nunique())
multi_type_directors = both_types[both_types == 2].index
multi_type_directors

In [ ]:
# Create director_type (movies + tv shows together)

director_type = (data_dir[data_dir['director'].isin(multi_type_directors)].groupby(['director', 'type']).size().reset_index(name='count'))

In [ ]:
# Fetch top 10 directors who did both movies and tv shows

top_directors_both = (data_dir[data_dir['director'].isin(multi_type_directors)]['director'].value_counts().head(10).index)

In [ ]:
# Filter director_type for top 10 rows

director_type = director_type[director_type['director'].isin(top_directors_both)]

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=director_type,y='director',x='count',hue='type')
plt.title("Directors Working in Both Movies & TV Shows")
plt.xlabel("Number of Titles")
plt.ylabel("Director")
plt.legend(title="Content Type")
plt.show()

**Note:** The above plot shows the top 10 directors who have worked on both Movies and TV Shows

### Comparison of tv shows vs. movies

In [ ]:
# Movies and TV Show Counts

type_counts = data['type'].value_counts()
type_counts

In [ ]:
# Movies and TV Show Percentage

perc_count_type = data['type'].value_counts(normalize=True) * 100
perc_count_type

In [ ]:
# Countplot for Content Types

plt.figure(figsize=(8, 6))
sns.countplot(data=data, x='type')

**Analysis:** The data is mainly divided into 2 types: Movies and TV Shows. Movies being 69.6% of total data and TV shows being 30.4% of total data (original dataset)

In [ ]:
# For new data set after handling missing values and un-nesting

type_counts_new = data_new['type'].value_counts()
type_counts_new

In [ ]:
perc_count_type_new = data_new['type'].value_counts(normalize=True) * 100
perc_count_type_new

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='type', y='release_year', data = data_new)
plt.title("Movies vs TV Shows")
plt.show()

**Analysis:** TV Shows were started after movies. Movies have a wide spread over the years which means the Netflix has more older movies than TV shows. However, in recent years Netflix has been increasingly investing in TV shows, suggesting a strategic shift toward show based content.

#### Tv shows vs Movies: Release year wise

In [ ]:
# Movies count per year

movies_per_year = data_new[data_new['type'] == 'Movie'].groupby('release_year').size()
movies_per_year

In [ ]:
# TV Shows count per year

tv_per_year = data_new[data_new['type'] == 'TV Show'].groupby('release_year').size()
tv_per_year

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(movies_per_year.index, movies_per_year.values, label="Movies")
plt.plot(tv_per_year.index, tv_per_year.values, label="TV Shows")
plt.title("Movies vs TV Shows Released Over Time")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.legend()
plt.show()

**Analysis:** Movies were dominant in earlier years (before 2015). TV Shows have increased significantly after 2015 - 2016. This also implies that Netlix wants to priotize the tv show content to keep users engaged for longer durations, thus improving retention and subscriptions continuity.

#### Tv shows vs Movies: Duration wise

In [ ]:
# Extract numeric duration for Movies
data_new['duration_minutes'] = pd.to_numeric(data_new.loc[data_new['type'] == 'Movie', 'duration'].str.replace(' min', '', regex=False), errors='coerce')

# Extract number of seasons for TV Shows
data_new['duration_seasons'] = pd.to_numeric(data_new.loc[data_new['type'] == 'TV Show', 'duration'].str.split(' ').str[0], errors='coerce')

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data_new['duration_minutes'].dropna(), bins=30)
plt.title("Distribution of Movie Duration (Minutes)")
plt.xlabel("Minutes")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data_new['duration_seasons'].dropna(), bins=10)
plt.title("Distribution of TV Show Seasons")
plt.xlabel("Number of Seasons")
plt.ylabel("Count")
plt.show()

**Analysis:** Movies mostly range between 80-150 minutes. TV shows typically range from 1-3 seasons, with fewer long-running series

#### Tv shows vs Movies: Genre wise

In [ ]:
# Top 10 genres overall

top_10_genres = data_new['listed_in'].value_counts().head(10).index

In [ ]:
genre_type = (data_new.groupby(['listed_in', 'type']).size().reset_index(name='count'))

genre_type = genre_type[genre_type['listed_in'].isin(top_10_genres)]

plt.figure(figsize=(10,6))
sns.barplot(data=genre_type, y='listed_in', x='count', hue='type')
plt.title("Top 10 Genres: Movies vs TV Shows")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.show()

**Anslysis:** Movies on Netflix are more of drama, comedy, action genre, whereas TV shows are more of drama and international content. This suggests that Netflix uses TV shows to cater to global audiences.

#### Tv shows vs Movies: Country wise

In [ ]:
# Top 10 countries (excluding Unknown)
top_countries = (data_new[data_new['country'] != 'Unknown Country']['country'].value_counts().head(10).index)

country_type = (data_new[data_new['country'].isin(top_countries)].groupby(['country', 'type']).size().reset_index(name='count'))

plt.figure(figsize=(10,6))
sns.barplot(data=country_type, y='country', x='count', hue='type')
plt.title("Top 10 Countries: Movies vs TV Shows")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.show()

In [ ]:
# Filtering only Movies
movies_df = data_new[(data_new['type'] == 'Movie') & (data_new['country'] != 'Unknown Country')]

# Group by country and counting unqique movie titles
movies_by_country = (movies_df.groupby('country')['title'].nunique().sort_values(ascending=False).head(10))

In [ ]:
plt.figure(figsize=(8,5))
movies_by_country.plot(kind='barh')
plt.xlabel("Number of Unique Movies")
plt.ylabel("Country")
plt.title("Top 10 Countries Producing Movies on Netflix")
plt.gca().invert_yaxis()   # so highest is on top
plt.show()

In [ ]:
# Filtering only TV Shows
tv_df = data_new[(data_new['type'] == 'TV Show') & (data_new['country'] != 'Unknown Country')]

# Group by country and counting unique TV show titles
tv_by_country = (tv_df.groupby('country')['title'].nunique().sort_values(ascending=False).head(10))

In [ ]:
plt.figure(figsize=(8,5))
tv_by_country.plot(kind='barh')
plt.xlabel("Number of Unique TV Shows")
plt.ylabel("Country")
plt.title("Top 10 Countries Producing TV Shows on Netflix")
plt.gca().invert_yaxis()
plt.show()

Movies are heavily produced in: USA, India, UK

TV Shows have stronger representation from USA, India, Japan, South Korea and other international markets

#### Tv shows vs Movies: Rating wise

In [ ]:
# Top 10 ratings
top_10_ratings = data_new['rating'].value_counts().head(10).index

rating_type = (data_new[data_new['rating'].isin(top_10_ratings)].groupby(['rating', 'type']).size().reset_index(name='count'))

plt.figure(figsize=(10,6))
sns.barplot(data=rating_type,x='rating',y='count', hue='type')
plt.xticks(rotation=45)
plt.title("Top 10 Ratings: Movies vs TV Shows")
plt.xlabel("Rating")
plt.ylabel("Count of Movies and TV Shows")
plt.show()

**Analysis:** Ratings for movies: TV-MA, TV-14, R. Ratings for TV shows: TV-MA, TV-14, TV-PG. This indicates that Netflix's movies tend to be slightly more mature themed, whereas TV shows cater to a slightly broader audience.

## Best time to launch TV show/Movie
### Best month to launch

In [ ]:
# Creating new columns month_added and month_name for further analysis

data_new['month_added'] = data_new['date_added'].dt.month
data_new['month_name'] = data_new['date_added'].dt.month_name()

In [ ]:
# Filter only TV shows data

tv_shows = data_new[data_new['type'] == 'TV Show'].copy()

In [ ]:
# Filter only movies data

movies_data = data_new[data_new['type'] == 'Movie'].copy()

In [ ]:
tv_month_counts = tv_shows['month_name'].value_counts().reindex(['January','February','March','April','May','June','July','August','September','October','November','December'])

print("TV Shows by Month:\n", tv_month_counts)

In [ ]:
movie_month_counts = movies_data['month_name'].value_counts().reindex(['January','February','March','April','May','June','July','August','September','October','November','December'])

print("\nMovies by Month:\n", movie_month_counts)

In [ ]:
# TV Shows

tv_month_counts.plot(kind='bar', figsize=(8,5))
plt.title("TV Shows Added by Month")
plt.ylabel("Count")
plt.xlabel("Month")
plt.xticks(rotation=45)
plt.show()

# Movies

movie_month_counts.plot(kind='bar', figsize=(8,5))
plt.title("Movies Added by Month")
plt.ylabel("Count")
plt.xlabel("Month")
plt.xticks(rotation=45)
plt.show()

**Analysis:** Peak Months for TV Shows: July, September, December. Best time to launch for movies: July

### Best week to launch

In [ ]:
# Adding a new column week_added

data_new['week_added'] = data_new['date_added'].dt.isocalendar().week

In [ ]:
tv_shows = data_new[data_new['type'] == 'TV Show']
movies = data_new[data_new['type'] == 'Movie']

In [ ]:
# TV Shows

tv_week_counts = tv_shows.groupby('week_added').size().sort_values(ascending=False)
print("Top 10 weeks for TV Shows:\n", tv_week_counts.head(10))

In [ ]:
# Movies

movie_week_counts = movies.groupby('week_added').size().sort_values(ascending=False)
print("\nTop 10 weeks for Movies:\n", movie_week_counts.head(10))

In [ ]:
# TV Shows

tv_week_counts.sort_index().plot(kind='line', figsize=(10,5))
plt.title("TV Shows Released by Week")
plt.xlabel("Week of Year")
plt.ylabel("Count")
plt.show()

# Movies

movie_week_counts.sort_index().plot(kind='line', figsize=(10,5))
plt.title("Movies Released by Week")
plt.xlabel("Week of Year")
plt.ylabel("Count")
plt.show()

**Analysis:** Best week to launch TV shows is between week 37-39 (mid to late September) and for movies week 27-30 (July / early August). Reason may vary from country to country. In US for movies, holidays are going on during July, so this is the best time to launch movies. For tv shows during September, people spend more time at home as weather changes.

## Actors/directors of different types of shows/movies

In [ ]:
# Group by actor and counting unique titles

top_actors = (data_new[data_new['cast'] != 'Unknown Cast'].groupby('cast')['title'].nunique().sort_values(ascending=False).head(10))

In [ ]:
top_actors.plot(kind='bar', figsize=(10,5))
plt.title("Top 10 Actors by Number of Unique Titles")
plt.ylabel("Number of Titles")
plt.xlabel("Actor Name")
plt.xticks(rotation=45, ha='right')
plt.show()

**Analysis:** Most number of distinct Netflix titles feature mainly Indian actors. This demonstrates Netflix's significant presence of Indian content. Netflix frequently acquires several films with well-known actors. Bollywood actors in particular have a significant impact on Netflix's content.

In [ ]:
# Group by director and counting unique titles

top_directors = (data_new[data_new['director'] != 'Unknown Director'].groupby('director')['title'].nunique().sort_values(ascending=False).head(10))

In [ ]:
top_directors.plot(kind='bar', figsize=(8,5))
plt.title("Top 10 Directors by Number of Unique Titles (Excluding Unknown)")
plt.ylabel("Number of Titles")
plt.xlabel("Director Name")
plt.xticks(rotation=45, ha='right')
plt.show()

**Analysis:** Rajiv Chilaka leads because of multiple titles. This implies that Netflix frequently collaborates with directors who consistently produce work that appeals to audiences. Directors with a track record of success and a large volume of content are given preference by Netflix.

In [ ]:
# Combine all genres into one single string
all_genres = " ".join(movies_data['listed_in'].dropna())

# Replace commas with space so WordCloud treats each genre separately
all_genres = all_genres.replace(",", "")

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_genres)

plt.figure(figsize=(8,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Most Produced Movie Genres on Netflix")
plt.show()

**Analysis:** From the WordCloud and frequency counts, we can interpret that the most dominant genres for Movies are- International Movies, Dramas, Comedies, Action & Adventure, Independent Movies

In [ ]:
# Split genres and count frequency
genre_counts = (movies_data['listed_in'].str.split(', ').explode().value_counts())

In [ ]:
genre_counts.head(10).plot(kind='bar', figsize=(8,5))
plt.title("Top 10 Movie Genres on Netflix")
plt.ylabel("Number of Movies")
plt.xlabel("Genre")
plt.xticks(rotation=45, ha='right')
plt.show()

**Analysis:** International Movies appear the largest which means Netflix has a very global content library. Dramas are highly frequent they appeal to a wide audience across countries. Comedies and Action are also prominent indicating Netflix balances serious and entertaining content. Netflix produces/acquires more international and drama-based movies, showing a strategy focused on global reach.

## Days taken to release movies/tv shows

In [ ]:
# Converting date_added to datetime format

movies_data['date_added'] = pd.to_datetime(movies_data['date_added'])

In [ ]:
# Creating an approximate release date: June 30 of release_year. This is an approximation since only the release year is available.

movies_data['approx_release_date'] = pd.to_datetime(movies_data['release_year'].astype(str) + '-06-30')

In [ ]:
# Computing difference in days

movies_data['days_to_netflix'] = (movies_data['date_added'] - movies_data['approx_release_date']).dt.days

In [ ]:
# Checking the most common days difference

mode_days = movies_data['days_to_netflix'].mode()[0]
print("Most common number of days before adding to Netflix:", mode_days)

In [ ]:
movies_data['days_to_netflix'].hist(bins=30, figsize=(8,5))
plt.title("Distribution of Days Between Release and Netflix Addition")
plt.xlabel("Days")
plt.ylabel("Number of Movies")
plt.show()

**Analyis:** The most common time gap between a movie's release and its addition to Netflix is around 3 years and 3 months. Netflix often acquires movies well after their initial theatrical run. Many movies in the catalog are older licensed films, not fresh releases. One of the possible reasons for such delay can be prioritizing content whose licensing cost has decreased over time. Other can be regional or international films that arrive later.

## Number of movies released per year

In [ ]:
movies_per_year = (movies_data.groupby('release_year').size().reset_index(name='movie_count'))

In [ ]:
recent_movies = movies_per_year[movies_per_year['release_year'] >= 1995]

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(recent_movies['release_year'], recent_movies['movie_count'])
plt.title("Number of Movies Released Per Year (Last 20-30 Years)")
plt.xlabel("Release Year")
plt.ylabel("Number of Movies")
plt.grid(True)
plt.show()

**Analysis:** The number of movies added per year from 1995 to 2010 is relatively low and fairly stable. This means that Netflix initially focused more on TV shows or licensed content, not on new movies. After 2015-2019, there is a sharp increase in the number of movies released per year. This can be due to either investing in original movies, expansion into global markets. The number of movies per year reaches its highest point around 2019-2020. This corresponds to Netflix's major push into original content and rise of tv shows over movies. After 2020, there was a drop. This may be due to COVID-19 and more focus on tv shows instead of movies.

## TV Shows vs movies in recent years

In [ ]:
# Count number of Movies and TV Shows per year
content_per_year = (data_new.groupby(['release_year', 'type']).size().reset_index(name='count'))

# Pivot for easier comparison
content_pivot = content_per_year.pivot(index='release_year', columns='type', values='count')

In [ ]:
# Recent trend data analysis (after 2013)

recent_trends = content_pivot[content_pivot.index >= 2013]

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(recent_trends.index, recent_trends['Movie'], label='Movies')
plt.plot(recent_trends.index, recent_trends['TV Show'], label='TV Shows')

plt.title("Movies vs TV Shows Released Per Year (Recent Years)")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
recent_trends['total'] = recent_trends['Movie'] + recent_trends['TV Show']
recent_trends['movie_pct'] = (recent_trends['Movie'] / recent_trends['total']) * 100
recent_trends['tv_pct'] = (recent_trends['TV Show'] / recent_trends['total']) * 100

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(recent_trends.index, recent_trends['movie_pct'], label='Movies %')
plt.plot(recent_trends.index, recent_trends['tv_pct'], label='TV Shows %')

plt.title("Netflix Focus: Movies vs TV Shows (%) in Recent Years")
plt.xlabel("Release Year")
plt.ylabel("Percentage Share")
plt.legend()
plt.grid(True)
plt.show()

**Analysis:** It can be observed that the movies consistently outnumber TV shows in most years. However, the gap has been narrowing in recent years. 2013-2016 was a movie dominant phase. Netflix released far more movies than TV shows. From 2017-2019, tv shows begin catching up. Netflix started investing more in original tv content. From 2020 inwards stronger focus was on tv shows. TV Shows growing at a faster rate than movies. Percentage share of TV Shows increasing year by year. This may be due to possibility of higher subscription for tv shows compared to movies. COVID might have boosted demand for long-form content.

## Understanding content availability for different countries

In [ ]:
content_by_country = (data_new.groupby('country')['title'].nunique().sort_values(ascending=False))

In [ ]:
content_by_country.head(15).plot(kind='bar', figsize=(10,5))
plt.title("Top 15 Countries by Number of Titles on Netflix")
plt.ylabel("Number of Titles")
plt.xlabel("Country")
plt.xticks(rotation=45, ha='right')
plt.show()

**Analysis:** Netflix has the largest catalog of US-based content.Netflix has invested heavily in Bollywood and regional Indian content. Inclusion of UK, Japan, South Korea, Spain, France shows strong global diversity. This indicates Netflix is not just US-centric but actively acquiring international content.

In [ ]:
country_type = (data_new.groupby(['country','type']).size().reset_index(name='count'))

# Pivot for easier comparison
country_pivot = country_type.pivot(index='country', columns='type', values='count')

In [ ]:
top10 = country_pivot.sort_values('Movie', ascending=False).head(10)

top10.plot(kind='bar', stacked=True, figsize=(10,5))
plt.title("Movies vs TV Shows by Country (Top 10)")
plt.ylabel("Number of Titles")
plt.xlabel("Country")
plt.xticks(rotation=45, ha='right')
plt.show()

Analysis: From the plot, it can be observed that USA and India has more movies than tv shows. South Korea and UK have a more balanced mix. Japan have a higher proportion of TV Shows. Spain and France have a mix of both, but leaning slightly toward movies. Countries like Japan and South Korea contribute many TV series i.e. strong serial content culture.

# Conclusion
## Content Distribution by Country (Movies vs TV Shows)

1. USA and India produce more movies than tv shows. This indicates a stronger tradition/industry focus on film production compared to serialized content.
2. South Korea and the UK have a more balanced mix between Movies and TV Shows. This suggests a healthy balance for both cinematic and episodic storytelling.
3. Japan has a higher proportion of tv shows than movies. It can be included that Japanese people prefer more of the serialized content (anime, dramas, variety shows, etc.).
4. Spain and France appear to have a mix of both, but slightly leaning toward movies. This indicates a mixed but somewhat film-centric content.

## High Level Detail

Netflix's content library reflects regional content cultures:

- Film-dominant markets (USA, India)
- Series-dominant markets (Japan)
- Hybrid/balanced markets (UK, South Korea, Spain, France)

## Recommendation

1. Prioritize TV series investments in Japan
2. Continue focusing more on films in the USA and India. However, Netflix can invest more in original series in India to reduce over-reliance on movies.
3. Maintain a mixed strategy in the UK and South Korea
4. Reduce the delay between theatrical/TV release and Netflix availability. Especially in markets like India where competition is high.
5. Netflix should keep investing in international dramas, crime series.
6. Netflix should increase TV-PG and TV-G content for kids and family-friendly viewing.
7. Netflix's content is highly US-centric. Non-English markets have far fewer titles available. To grow internationally, Netflix should increase the number of titles in Japan, South Korea, Spain, Mexico.